In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.optim as optim
from torchvision.models import vgg16

In [ ]:
transform = transforms.Compose([
        transforms.Resize(256),
        transforms.RandomCrop(256),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])


class DTD_Dataset(Dataset):

    def __init__(self, root, file_list, transform=None):
        self.root = root
        self.transform = transform

        with open(file_list, mode='r', encoding='utf-8') as f:
            files = f.read()
            self.files = [line for line in files.splitlines() if line.strip()]

        unique_classes = sorted(list(set([line.split('/')[0] for line in self.files])))
        self.class_to_idx = {class_name: i for i, class_name in enumerate(unique_classes)}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        relative_path = self.files[idx]
        image_path = os.path.join(self.root, relative_path)
        img = Image.open(image_path).convert('RGB')
        string_label, _ = relative_path.split('/')
        label = self.class_to_idx[string_label]

        if self.transform:
            img = self.transform(img)

        return img, label

In [ ]:
path_images = "drive/MyDrive/DeepLearning/dtd/images"
path_labels = "drive/MyDrive/DeepLearning/dtd/labels"
train_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "train1.txt"), transform)
val_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "val1.txt"), transform)
test_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "test1.txt"), transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, hidden_dim=128, embedding_dim=256):
        super().__init__()
        self.conv1 = nn.Conv2d(3, hidden_dim // 2, kernel_size=4, stride=2, padding=1)              # 256x256 -> 128x128
        self.conv2 = nn.Conv2d(hidden_dim // 2, hidden_dim, kernel_size=4, stride=2, padding=1)     # 128x128 -> 64x64
        self.downsample = nn.Conv2d(hidden_dim, embedding_dim, kernel_size=4, stride=4, padding=0)  # 64x64 -> 16x16

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.downsample(x)
        return x

class Decoder(nn.Module):
    def __init__(self, embedding_dim=256, hidden_dim=128):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(embedding_dim, hidden_dim, kernel_size=4, stride=4, padding=0)   #16x16 -> 64x64
        self.conv1 = nn.ConvTranspose2d(hidden_dim, hidden_dim // 2, kernel_size=4, stride=2, padding=1)    # 64x64 -> 128x128
        self.conv2 = nn.ConvTranspose2d(hidden_dim // 2, 3, kernel_size=4, stride=2, padding=1)     # 128x128 -> 256x256

    def forward(self, x):
        x = self.upsample(x)
        x = F.relu(self.conv1(x))
        x = self.conv2(x)
        x = torch.tanh(x)
        return x

class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings=1024, embedding_dim=256, commitment_cost=0.25):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost

        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        self.embeddings.weight.data.uniform_(-1/self.num_embeddings, 1/self.num_embeddings)

    def forward(self, z):
        z_flattened = z.permute(0, 2, 3, 1).contiguous()
        z_flattened = z_flattened.view(-1, self.embedding_dim)

        distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                     + torch.sum(self.embeddings.weight**2, dim=1)
                     - 2 * torch.matmul(z_flattened, self.embeddings.weight.t()))

        encoding_indices = torch.argmin(distances, dim=1)
        encodings = F.one_hot(encoding_indices, self.num_embeddings).float()

        quantized = torch.matmul(encodings, self.embeddings.weight)
        quantized = quantized.view(z.shape[0], z.shape[2], z.shape[3], self.embedding_dim)
        quantized = quantized.permute(0, 3, 1, 2).contiguous()

        e_latent_loss = F.mse_loss(quantized.detach(), z)
        q_latent_loss = F.mse_loss(quantized, z.detach())
        loss = q_latent_loss + self.commitment_cost * e_latent_loss

        quantized = z + (quantized - z).detach()

        return quantized, loss


VQGAN:
- Generator: Encoder-VQ-Decoder. It takes a real texture image and tries to generate a flawless, high-resolution copy of it
- Discriminator: it looks at the original image and the generated copy, trying to figure out which one is the fake one

Perceptual Loss (Reconstruction Loss): LPIPS (Learned Perceptual Image Patch Similarity). It does not look at the raw pixel coords, but the real texture and the generated texture through a pre-trained net that knows how to recognize shapes, edges and textures. It compares the hidden feature layers (it looks at the same freq of lines, same style of roughness, etc.)

VGG16: we freeze weights because when Perceptual Loss is high and the gradient flows backwards, only the Generator is updates. It's a pretrained net, we are only using its specialized vision

In [ ]:
class VQGAN(nn.Module):
    def __init__(self, hidden_dim=128, embedding_dim=256, num_embeddings=1024, commitment_cost=0.25):
        super().__init__()
        self.encoder = Encoder(hidden_dim, embedding_dim)
        self.vq = VectorQuantizer(num_embeddings, embedding_dim, commitment_cost)
        self.decoder = Decoder(embedding_dim, hidden_dim)

    def forward(self, x):
        z = self.encoder(x)
        quantized, vq_loss = self.vq(z)
        x_recon = self.decoder(quantized)
        return x_recon, vq_loss

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.conv1 = nn.Conv2d(3, hidden_dim, kernel_size=4, stride=2, padding=1)                   # (3, 256, 256) -> (64, 128, 128)
        self.relu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

        self.conv2 = nn.Conv2d(hidden_dim, hidden_dim * 2, kernel_size=4, stride=2, padding=1)      # (64, 128, 128) -> (128, 64, 64)
        self.norm1 = nn.BatchNorm2d(hidden_dim * 2)

        self.conv3 = nn.Conv2d(hidden_dim * 2, hidden_dim * 4, kernel_size=4, stride=2, padding=1)  # (128, 64, 64) -> (256, 32, 32)
        self.norm2 = nn.BatchNorm2d(hidden_dim * 4)

        self.conv4 = nn.Conv2d(hidden_dim * 4, 1, kernel_size=4, stride=1, padding=1)               # (256, 32, 32) -> (1, 31, 31)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.norm1(self.conv2(x)))
        x = self.relu(self.norm2(self.conv3(x)))
        x = self.conv4(x)
        return x

In [ ]:
class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        VGG16 = vgg16(weights='DEFAULT').features
        self.slice = nn.Sequential(*list(VGG16.children())[:16]).eval()
        for param in self.slice.parameters():
            param.require_grad = False

    def forward(self, real, fake):
        return F.mse_loss(self.slice(fake), self.slice(real))

def calculate_adaptive_weight(recon_loss, g_loss, last_layer_weights):
    recon_grads = torch.autograd.grad(recon_loss, last_layer_weights, retain_graph=True)[0]
    g_grads = torch.autograd.grad(g_loss, last_layer_weights, retain_graph=True)[0]

    lambda_weight = torch.norm(recon_grads) / (torch.norm(g_grads) + 1e-4)
    lambda_weight = torch.clamp(lambda_weight, 0.0, 1e4).detach()
    return lambda_weight

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = VQGAN().to(device)
discriminator = Discriminator().to(device)
optimizer_generator = optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.9))
optimizer_discriminator = optim.Adam(discriminator.parameters(), lr=5e-5, betas=(0.5, 0.9))
perceptual_loss_fn = PerceptualLoss().to(device)

In [ ]:
def train_step(generator, discriminator, perceptual_loss_fn, data, optimizer_g, optimizer_d, disc_start_step, current_global_step):
    # GENERATOR
    optimizer_g.zero_grad()
    recon_batch, vq_loss = generator(data)

    recon_loss = F.mse_loss(recon_batch, data)
    percept_loss = perceptual_loss_fn(recon_batch, data)

    fake_outputs = discriminator(recon_batch)
    g_loss = F.binary_cross_entropy_with_logits(fake_outputs, torch.ones_like(fake_outputs))

    # dynamic adaptive weight scheduling
    if current_global_step >= disc_start_step:
        last_layer_weights = generator.decoder.conv2.weight
        disc_weight = calculate_adaptive_weight(recon_loss, g_loss, last_layer_weights)
    else: disc_weight = 0.0

    total_g_loss = vq_loss + recon_loss + percept_loss + (0.8 * disc_weight * g_loss)
    total_g_loss.backward()
    optimizer_g.step()

    # DISCRIMINATOR
    total_d_loss = torch.tensor(0.0, device=data.device)
    if current_global_step >= disc_start_step:
        optimizer_d.zero_grad()

        real_outputs = discriminator(data)
        loss_real = F.binary_cross_entropy_with_logits(real_outputs, torch.ones_like(real_outputs))

        fake_outputs_d = discriminator(recon_batch.detach())
        loss_fake = F.binary_cross_entropy_with_logits(fake_outputs_d, torch.zeros_like(fake_outputs_d))

        total_d_loss = 0.5 * loss_real + 0.5 * loss_fake
        total_d_loss.backward()
        optimizer_d.step()

    return total_g_loss.item(), total_d_loss.item()


In [ ]:
global_step = 0
disc_start_step = 300
epochs = 10
for epoch in range(epochs):
    generator.train()
    discriminator.train()

    total_loss_generator = 0
    total_loss_discriminator = 0

    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        #print("step = ", global_step)
        g_loss_val, d_loss_val = train_step(
            generator, discriminator, perceptual_loss_fn, data,
            optimizer_generator, optimizer_discriminator,
            disc_start_step, global_step)

        global_step += 1
        total_loss_generator += g_loss_val
        total_loss_discriminator += d_loss_val

        #if batch_idx % 100 == 0:
        #    print(f'Epoch {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}]'
        #          f"| G-Loss: {g_loss_val:.4f} | D-Loss: {d_loss_val:.4f} | Step: {global_step}")

    avg_g_loss = total_loss_generator / len(train_loader)
    avg_d_loss = total_loss_discriminator / len(train_loader)
    print(f"====> Epoch {epoch} Finished | Avg G-Loss: {avg_g_loss:.4f} | Avg D-Loss: {avg_d_loss:.4f}\n")